# Experiment 3: Decision trees, pruning, and overfitting

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import load_iris, make_classification
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier, plot_tree

iris = load_iris()
X_tr, X_te, y_tr, y_te = train_test_split(iris.data, iris.target, test_size=0.3, random_state=0)

# Unpruned tree
full = DecisionTreeClassifier(random_state=0).fit(X_tr, y_tr)
print("Unpruned  depth:", full.get_depth(), " leaves:", full.get_n_leaves(),
      " test acc:", round(full.score(X_te, y_te), 3))

plt.figure(figsize=(14, 7))
plot_tree(full, feature_names=iris.feature_names, class_names=iris.target_names, filled=True)
plt.title("Unpruned decision tree (Iris)")
plt.show()

In [ ]:
# Depth control (pre-pruning)
for d in [1, 2, 3, 4]:
    t = DecisionTreeClassifier(max_depth=d, random_state=0).fit(X_tr, y_tr)
    print(f"max_depth={d}: leaves={t.get_n_leaves()}, test acc={t.score(X_te, y_te):.3f}")

# Cost-complexity pruning (post-pruning)
path = full.cost_complexity_pruning_path(X_tr, y_tr)
results = []
for a in path.ccp_alphas:
    t = DecisionTreeClassifier(ccp_alpha=a, random_state=0).fit(X_tr, y_tr)
    results.append((a, t.get_n_leaves(), t.score(X_te, y_te)))
best = max(results, key=lambda r: r[2])
print(f"Best ccp_alpha={best[0]:.4f}: leaves={best[1]}, test acc={best[2]:.3f}")

In [ ]:
# Overfitting on synthetic data: training accuracy keeps rising, test accuracy does not
Xs, ys = make_classification(n_samples=1000, n_features=20, n_informative=5, flip_y=0.1, random_state=1)
Xs_tr, Xs_te, ys_tr, ys_te = train_test_split(Xs, ys, test_size=0.3, random_state=1)

depths = list(range(1, 21))
tr_acc, te_acc = [], []
for d in depths:
    t = DecisionTreeClassifier(max_depth=d, random_state=0).fit(Xs_tr, ys_tr)
    tr_acc.append(t.score(Xs_tr, ys_tr))
    te_acc.append(t.score(Xs_te, ys_te))

plt.plot(depths, tr_acc, marker="o", label="train")
plt.plot(depths, te_acc, marker="o", label="test")
plt.xlabel("max_depth")
plt.ylabel("accuracy")
plt.title("Overfitting: train vs test accuracy")
plt.legend()
plt.show()